In [1]:
# Cell 1: Imports

from selenium import webdriver
from selenium.webdriver.chrome.service import Service 
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException

from bs4 import BeautifulSoup
import time
import json
from datetime import datetime, timedelta 
import re
import pandas as pd
import numpy as np 
import random

# For PostgreSQL
import psycopg2
from psycopg2 import sql 
from psycopg2.extras import execute_values 
from dotenv import load_dotenv # Để load file .env
import os # Để đọc biến môi trường

In [2]:
# Cell 2: Initialize WebDriver and Fetch First Page of VietnamWorks

VNW_BASE_URL = "https://www.vietnamworks.com/viec-lam?l=29&level=8" 
VNW_CURRENT_PAGE_HTML_OUTPUT = f"vnw_selenium_page_debug_{datetime.now().strftime('%Y%m%d_%H%M%S')}.html"

# Khai báo driver ở đây để Pylance biết nó tồn tại trong cell này
driver = None 
vnw_page_source = None 

# Kiểm tra xem driver có thể đã được khởi tạo từ lần chạy trước không (trong cùng session notebook)
_driver_temp = locals().get('driver', None)
if _driver_temp is not None:
    print("Sử dụng lại WebDriver đã được khởi tạo từ lần chạy trước của cell này hoặc cell khác.")
    driver = _driver_temp # Gán lại nếu nó đã tồn tại và có giá trị
else:
    print("Khởi tạo WebDriver mới cho VietnamWorks...")
    chrome_options_vnw = Options()
    chrome_options_vnw.add_argument("--no-sandbox")
    chrome_options_vnw.add_argument("--disable-dev-shm-usage")
    chrome_options_vnw.add_argument("--disable-gpu")
    chrome_options_vnw.add_argument("user-agent=Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/137.0.0.0 Safari/537.36") 
    chrome_options_vnw.add_argument("--window-size=1920,1080") 
    chrome_options_vnw.add_experimental_option("excludeSwitches", ["enable-automation"])
    chrome_options_vnw.add_experimental_option('useAutomationExtension', False)
    # chrome_options_vnw.add_argument("--headless") 

    try:
        driver = webdriver.Chrome(options=chrome_options_vnw) 
        print("WebDriver cho VietnamWorks đã khởi tạo thành công.")
    except Exception as e_init_driver:
        print(f"Lỗi khi khởi tạo WebDriver cho VietnamWorks: {e_init_driver}")
        driver = None 
 
if driver: 
    try:
        print(f"Truy cập VietnamWorks (Trang 1): {VNW_BASE_URL}")
        driver.get(VNW_BASE_URL)
        
        initial_page_load_wait = random.uniform(8, 12) 
        print(f"  Đợi {initial_page_load_wait:.1f} giây cho tải trang ban đầu...")
        time.sleep(initial_page_load_wait)
        
        print("  Bắt đầu quá trình cuộn trang để tải thêm jobs...")
        scroll_pause_time = random.uniform(2.5, 4.0) 
        last_height = driver.execute_script("return document.body.scrollHeight")
        scroll_attempts = 0
        max_scroll_attempts = 12 
        consecutive_no_change_scrolls = 0 

        while scroll_attempts < max_scroll_attempts:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(scroll_pause_time)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                consecutive_no_change_scrolls += 1
                if consecutive_no_change_scrolls >= 3: 
                    print("      Đã cuộn đến cuối hoặc không có nội dung mới. Dừng cuộn.")
                    break
            else:
                consecutive_no_change_scrolls = 0 
                last_height = new_height
            scroll_attempts += 1
        print("  Hoàn tất quá trình cuộn trang.")

        final_wait_after_scroll = random.uniform(7, 12) 
        print(f"  Đợi cuối cùng {final_wait_after_scroll:.1f} giây...")
        time.sleep(final_wait_after_scroll)
        
        print("  Đang lấy mã nguồn HTML của trang (page_source)...")
        vnw_page_source = driver.page_source # Gán giá trị cho vnw_page_source
        
        if vnw_page_source and len(vnw_page_source) > 50000: 
            with open(VNW_CURRENT_PAGE_HTML_OUTPUT, "w", encoding="utf-8") as f:
                f.write(vnw_page_source)
            print(f"  HTML của VietnamWorks đã lưu vào: {VNW_CURRENT_PAGE_HTML_OUTPUT} (Kích thước: {len(vnw_page_source)} bytes)")
        elif vnw_page_source:
            print(f"  CẢNH BÁO: HTML VNW nhỏ ({len(vnw_page_source)} bytes). Kiểm tra file debug.")
            with open(VNW_CURRENT_PAGE_HTML_OUTPUT, "w", encoding="utf-8") as f:
                f.write(vnw_page_source)
            print(f"  File HTML kích thước nhỏ đã lưu vào: {VNW_CURRENT_PAGE_HTML_OUTPUT}")
        else:
            print("  Không lấy được page_source từ VietnamWorks.")
            
    except TimeoutException: 
        print("Lỗi: Quá thời gian chờ khi tải trang VietnamWorks.")
    except Exception as e_vnw_fetch:
        print(f"Lỗi không xác định khi scrape VNW: {e_vnw_fetch}")
else:
    print("Lỗi: WebDriver không được khởi tạo. Không thể scrape VietnamWorks.")

# KHÔNG driver.quit() ở đây

Khởi tạo WebDriver mới cho VietnamWorks...
WebDriver cho VietnamWorks đã khởi tạo thành công.
Truy cập VietnamWorks (Trang 1): https://www.vietnamworks.com/viec-lam?l=29&level=8
  Đợi 8.6 giây cho tải trang ban đầu...
  Bắt đầu quá trình cuộn trang để tải thêm jobs...
      Đã cuộn đến cuối hoặc không có nội dung mới. Dừng cuộn.
  Hoàn tất quá trình cuộn trang.
  Đợi cuối cùng 7.6 giây...
  Đang lấy mã nguồn HTML của trang (page_source)...
  HTML của VietnamWorks đã lưu vào: vnw_selenium_page_debug_20250529_132405.html (Kích thước: 662079 bytes)


In [3]:
# Cell 3: Parse VietnamWorks HTML and Extract Job Data

vnw_page_source = locals().get('vnw_page_source', None) # Để Pylance biết
extracted_vnw_jobs_from_current_page = []

if vnw_page_source and len(vnw_page_source) > 30000: # Kiểm tra lại vnw_page_source
    print(f"Đang parse HTML của VietnamWorks (kích thước: {len(vnw_page_source)} bytes)...")
    soup_vnw = BeautifulSoup(vnw_page_source, 'html.parser')
    overall_job_items = soup_vnw.find_all('div', class_='view_job_item')
    
    if not overall_job_items: print("CẢNH BÁO: Không tìm thấy job items nào với class 'view_job_item'.")
    print(f"Tìm thấy {len(overall_job_items)} khối 'view_job_item' trên trang này.")

    current_scrape_date_vnw = datetime.now().strftime('%Y-%m-%d')
    current_scrape_datetime_vnw = datetime.now().isoformat()

    for item_index, overall_item in enumerate(overall_job_items):
        vnw_job_data = {
            'job_title': None, 'job_url': None, 'company_name': None,
            'salary_raw': None, 'location_raw': None, 
            'city_main': None, 'district': None, 
            'experience': "Thực tập sinh/Sinh viên", 
            'experience_standardized': "Intern/Fresher", 
            'post_date_raw': None, 'keywords': [], 
            'source_website': 'VietnamWorks',
            'scrape_date': current_scrape_date_vnw,
            'scraped_at_timestamp': current_scrape_datetime_vnw
        }
        
        try: 
            job_anchor_div = overall_item.find('div', attrs={'data-cname': 'preview-job'})
            if job_anchor_div:
                title_h2 = job_anchor_div.find('h2')
                if title_h2:
                    title_a_tag = title_h2.find('a')
                    if title_a_tag:
                        vnw_job_data['job_url'] = title_a_tag.get('href')
                        if vnw_job_data['job_url'] and not vnw_job_data['job_url'].startswith('http'):
                            vnw_job_data['job_url'] = "https://www.vietnamworks.com" + vnw_job_data['job_url']
                        full_title_text = title_a_tag.get_text(separator=' ', strip=True)
                        new_label_span = title_a_tag.find('span')
                        if new_label_span and "mới" in new_label_span.get_text(strip=True).lower():
                            vnw_job_data['job_title'] = full_title_text.replace(new_label_span.get_text(strip=True), "").strip()
                        else: vnw_job_data['job_title'] = full_title_text
            
            company_links = overall_item.find_all('a', href=re.compile(r'/nha-tuyen-dung/'), title=True)
            if company_links:
                for link in company_links: 
                    company_name_from_title = link.get('title')
                    if company_name_from_title: vnw_job_data['company_name'] = company_name_from_title; break 
                    elif link.get_text(strip=True): vnw_job_data['company_name'] = link.get_text(strip=True); break
            
            salary_location_block = overall_item.find('div', class_=re.compile(r'sc-bwGlVi\s+guifwN'))
            if salary_location_block:
                spans_in_block = salary_location_block.find_all('span', recursive=False)
                if len(spans_in_block) >= 1: vnw_job_data['salary_raw'] = spans_in_block[0].get_text(strip=True)
                if len(spans_in_block) >= 2:
                    first_span = spans_in_block[0]; divider_div = first_span.find_next_sibling('div')
                    if divider_div:
                        location_span = divider_div.find_next_sibling('span')
                        if location_span and location_span == spans_in_block[1]: vnw_job_data['location_raw'] = location_span.get_text(strip=True)
                        elif len(spans_in_block) == 2: vnw_job_data['location_raw'] = spans_in_block[1].get_text(strip=True)
                    elif len(spans_in_block) == 2: vnw_job_data['location_raw'] = spans_in_block[1].get_text(strip=True)
            elif not vnw_job_data['salary_raw']: 
                potential_sl_blocks = overall_item.find_all('div', class_=re.compile(r'^sc-bwGlVi'))
                for block_candidate in potential_sl_blocks:
                    spans_in_candidate = block_candidate.find_all('span', recursive=False)
                    if len(spans_in_candidate) >= 2:
                        first_span_text = spans_in_candidate[0].get_text(strip=True); second_span_text = spans_in_candidate[1].get_text(strip=True)
                        is_first_salary = '₫' in first_span_text or '$' in first_span_text or 'triệu' in first_span_text.lower() or any(kw in first_span_text.lower() for kw in ['thỏa thuận', 'thoa thuan', 'negotiable', 'thương lượng'])
                        is_second_location = not ('₫' in second_span_text or '$' in second_span_text or 'triệu' in second_span_text.lower() or any(kw in second_span_text.lower() for kw in ['thỏa thuận', 'thoa thuan', 'negotiable', 'thương lượng']))
                        if is_first_salary and is_second_location:
                            vnw_job_data['salary_raw'] = first_span_text; vnw_job_data['location_raw'] = second_span_text; break 
            
            if vnw_job_data['location_raw']:
                loc_raw_lower = vnw_job_data['location_raw'].lower()
                locations_list_vnw = [loc.strip() for loc in vnw_job_data['location_raw'].split(',')]
                if locations_list_vnw:
                    if "hồ chí minh" in loc_raw_lower or "hcm" in loc_raw_lower : vnw_job_data['city_main'] = "Hồ Chí Minh"
                    elif "hà nội" in loc_raw_lower: vnw_job_data['city_main'] = "Hà Nội"
                    elif "đà nẵng" in loc_raw_lower: vnw_job_data['city_main'] = "Đà Nẵng"
                    else: vnw_job_data['city_main'] = locations_list_vnw[0]
                vnw_job_data['district'] = None 

            date_keywords_block = overall_item.find('div', class_=re.compile(r'sc-bwGlVi\s+dFOVWC')) 
            if date_keywords_block:
                post_date_div = date_keywords_block.find('div', class_=re.compile(r'sc-fnLEGM')) 
                if post_date_div: vnw_job_data['post_date_raw'] = post_date_div.get_text(strip=True)
                keyword_labels = date_keywords_block.select('ul li label[title]') 
                for label in keyword_labels:
                    keyword = label.get('title')
                    if keyword and not keyword.startswith('+'): vnw_job_data['keywords'].append(keyword)

            if vnw_job_data.get('job_title') and vnw_job_data.get('job_url'):
                extracted_vnw_jobs_from_current_page.append(vnw_job_data)
            # else: print(f"  CẢNH BÁO: Bỏ qua item VNW {item_index + 1} thiếu Title/URL.") # Bỏ bớt log
        except Exception as e_parse_item_vnw:
            print(f"  LỖI parse item VNW {item_index + 1}: {e_parse_item_vnw}")
            continue
            
    print(f"\nĐã trích xuất {len(extracted_vnw_jobs_from_current_page)} jobs từ trang VietnamWorks này.")
    if extracted_vnw_jobs_from_current_page:
        print("\n--- Ví dụ 3 jobs đầu tiên từ trang VNW hiện tại ---")
        for i, job in enumerate(extracted_vnw_jobs_from_current_page[:3]):
            print(f"\n--- Job VNW (Raw) {i+1} ---"); [print(f"  {key.replace('_', ' ').title()}: {value}") for key, value in job.items()]
    else: print("Không trích xuất được job nào từ trang VNW này (sau khi lọc).")
else: print("Không có 'vnw_page_source' hợp lệ để parse. Chạy lại Cell 2.")

Đang parse HTML của VietnamWorks (kích thước: 662079 bytes)...
Tìm thấy 50 khối 'view_job_item' trên trang này.

Đã trích xuất 50 jobs từ trang VietnamWorks này.

--- Ví dụ 3 jobs đầu tiên từ trang VNW hiện tại ---

--- Job VNW (Raw) 1 ---
  Job Title: Marketing Communications Intern (B2C)
  Job Url: https://www.vietnamworks.com/marketing-communications-intern-b2c-1913016-jv?source=searchResults&searchType=2&placement=1913016&sortBy=date
  Company Name: Navigos Group
  Salary Raw: Tới 3tr ₫/tháng
  Location Raw: Hồ Chí Minh
  City Main: Hồ Chí Minh
  District: None
  Experience: Thực tập sinh/Sinh viên
  Experience Standardized: Intern/Fresher
  Post Date Raw: Hôm nay
  Keywords: ['Content Writing', 'Script Video', 'Tiktok']
  Source Website: VietnamWorks
  Scrape Date: 2025-05-29
  Scraped At Timestamp: 2025-05-29T13:24:46.253156

--- Job VNW (Raw) 2 ---
  Job Title: Thực Tập Sinh Quan Hệ Khách Hàng Doanh Nghiệp
  Job Url: https://www.vietnamworks.com/thuc-tap-sinh-quan-he-khach-hang-

In [4]:
# Cell 4: Save Extracted VietnamWorks Data to JSON File and Close WebDriver

VNW_JSON_OUTPUT_FILENAME = f"vnw_extracted_jobs_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
extracted_vnw_jobs_from_current_page = locals().get('extracted_vnw_jobs_from_current_page', []) # Để Pylance biết

if extracted_vnw_jobs_from_current_page: # Chỉ lưu nếu có dữ liệu
    try:
        print(f"\nĐang chuẩn bị lưu {len(extracted_vnw_jobs_from_current_page)} job items từ VietnamWorks vào file JSON...")
        with open(VNW_JSON_OUTPUT_FILENAME, 'w', encoding='utf-8') as f_json:
            json.dump(extracted_vnw_jobs_from_current_page, f_json, ensure_ascii=False, indent=4)
        print(f"Thành công! Dữ liệu VietnamWorks đã được lưu vào file: '{VNW_JSON_OUTPUT_FILENAME}'")
    except Exception as e_save_vnw:
        print(f"Lỗi khi lưu file JSON của VietnamWorks: {e_save_vnw}")
elif 'extracted_vnw_jobs_from_current_page' in locals() and not extracted_vnw_jobs_from_current_page:
     print("Không có dữ liệu job nào từ VietnamWorks để lưu (danh sách rỗng).")
else:
    print("Biến 'extracted_vnw_jobs_from_current_page' không tồn tại. Hãy đảm bảo Cell 3 đã chạy đúng.")

# Đóng WebDriver (Rất quan trọng!)
driver = locals().get('driver', None) # Lấy lại driver
if driver is not None:
    try:
        print("\nĐang đóng WebDriver (sau khi xử lý VietnamWorks)...")
        driver.quit()
        print("WebDriver đã được đóng thành công.")
        driver = None # QUAN TRỌNG: Gán lại là None sau khi đóng để các lần chạy cell sau biết
    except Exception as e_quit_vnw:
        print(f"Lỗi khi đóng WebDriver (sau khi xử lý VietnamWorks): {e_quit_vnw}")
else:
    print("\nWebDriver không tồn tại hoặc đã được đóng trước đó.")

print(f"\n--- HOÀN TẤT SCRAPING VÀ LƯU DỮ LIỆU CHO VIETNAMWORKS (File: {VNW_JSON_OUTPUT_FILENAME}) ---")


Đang chuẩn bị lưu 50 job items từ VietnamWorks vào file JSON...
Thành công! Dữ liệu VietnamWorks đã được lưu vào file: 'vnw_extracted_jobs_20250529_132446.json'

Đang đóng WebDriver (sau khi xử lý VietnamWorks)...
WebDriver đã được đóng thành công.

--- HOÀN TẤT SCRAPING VÀ LƯU DỮ LIỆU CHO VIETNAMWORKS (File: vnw_extracted_jobs_20250529_132446.json) ---


In [5]:
# Cell 5: Load VietnamWorks Data from JSON and Initial Processing

VNW_JSON_OUTPUT_FILENAME = locals().get('VNW_JSON_OUTPUT_FILENAME', None) # Lấy lại tên file
df_jobs_vnw = None # Khởi tạo để Pylance biết

if VNW_JSON_OUTPUT_FILENAME:
    print(f"Đang đọc dữ liệu VietnamWorks từ file JSON: {VNW_JSON_OUTPUT_FILENAME}...")
    try:
        df_jobs_vnw = pd.read_json(VNW_JSON_OUTPUT_FILENAME, encoding='utf-8') 
        if df_jobs_vnw.empty:
            print("!!! Cảnh báo: Không có dữ liệu nào được đọc từ file JSON của VietnamWorks.")
        else:
            print(f"Đã đọc thành công {len(df_jobs_vnw)} job items từ file JSON của VietnamWorks.")
            print("\nThông tin cơ bản về DataFrame VietnamWorks (df_jobs_vnw.info()): ")
            df_jobs_vnw.info()
            print('\n === 5 dòng dữ liệu đầu tiên từ VietnamWorks (df_jobs_vnw.head()) ===')
            display(df_jobs_vnw.head(5))
            print("\n--- Kiểm tra các giá trị bị thiếu (NaN) cho mỗi cột (VietnamWorks) ---")
            print(df_jobs_vnw.isnull().sum())
            # ... (Có thể thêm lại phần value_counts nếu bạn muốn xem xét ở đây) ...
            print("\nDataFrame 'df_jobs_vnw' đã sẵn sàng cho các bước xử lý chi tiết hơn.")
    except FileNotFoundError:
        print(f"Lỗi: Không tìm thấy file '{VNW_JSON_OUTPUT_FILENAME}'.")
    except Exception as e_read_vnw:
        print(f"Lỗi khi đọc hoặc xử lý file JSON của VietnamWorks: {e_read_vnw}")
else:
    print("Không có tên file JSON của VietnamWorks (VNW_JSON_OUTPUT_FILENAME). Hãy chạy lại Cell 4.")

Đang đọc dữ liệu VietnamWorks từ file JSON: vnw_extracted_jobs_20250529_132446.json...
Đã đọc thành công 50 job items từ file JSON của VietnamWorks.

Thông tin cơ bản về DataFrame VietnamWorks (df_jobs_vnw.info()): 
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50 entries, 0 to 49
Data columns (total 14 columns):
 #   Column                   Non-Null Count  Dtype  
---  ------                   --------------  -----  
 0   job_title                50 non-null     object 
 1   job_url                  50 non-null     object 
 2   company_name             50 non-null     object 
 3   salary_raw               50 non-null     object 
 4   location_raw             50 non-null     object 
 5   city_main                50 non-null     object 
 6   district                 0 non-null      float64
 7   experience               50 non-null     object 
 8   experience_standardized  50 non-null     object 
 9   post_date_raw            50 non-null     object 
 10  keywords                 50 

,job_title,job_url,company_name,salary_raw,location_raw,city_main,district,experience,experience_standardized,post_date_raw,keywords,source_website,scrape_date,scraped_at_timestamp
0,Marketing Communications Intern (B2C),https://www.vietnamworks.com/marketing-communi...,Navigos Group,Tới 3tr ₫/tháng,Hồ Chí Minh,Hồ Chí Minh,NaN,Thực tập sinh/Sinh viên,Intern/Fresher,Hôm nay,"[Content Writing, Script Video, Tiktok]",VietnamWorks,2025-05-29,2025-05-29T13:24:46.253156
1,Thực Tập Sinh Quan Hệ Khách Hàng Doanh Nghiệp,https://www.vietnamworks.com/thuc-tap-sinh-qua...,"Chailease International Leasing Co., Ltd. (Hea...",Tới $ 140 /tháng,"Hồ Chí Minh, Bình Dương, Đồng Nai",Hồ Chí Minh,NaN,Thực tập sinh/Sinh viên,Intern/Fresher,Cập nhật: 28/05/2025,"[Tìm Kiếm Khách Hàng, Khách Hàng Doanh Nghiệp]",VietnamWorks,2025-05-29,2025-05-29T13:24:46.253156
2,Internship Civil Design Engineer,https://www.vietnamworks.com/internship-civil-...,"Danieli – Industrielle Beteiligung Co., Ltd",Thương lượng,Hồ Chí Minh,Hồ Chí Minh,NaN,Thực tập sinh/Sinh viên,Intern/Fresher,Cập nhật: 27/05/2025,"[AutoCAD 2D, AVEVA E3D, EPLAN]",VietnamWorks,2025-05-29,2025-05-29T13:24:46.253156
3,Chăm Sóc Khách Hàng (Làm Ca Đêm - Thưởng Hấp D...,https://www.vietnamworks.com/cham-soc-khach-ha...,Fastboy Marketing,"$ 400-1,500 /tháng",Hồ Chí Minh,Hồ Chí Minh,NaN,Thực tập sinh/Sinh viên,Intern/Fresher,Cập nhật: 26/05/2025,"[Phiên Dịch, Customer Management]",VietnamWorks,2025-05-29,2025-05-29T13:24:46.253156
4,Sales Intern Part Time,https://www.vietnamworks.com/sales-intern-part...,"FUJIFILM Business Innovation Vietnam Co., Ltd",Thương lượng,"Hồ Chí Minh, Hà Nội",Hồ Chí Minh,NaN,Thực tập sinh/Sinh viên,Intern/Fresher,Cập nhật: 26/05/2025,"[B2B Sales, Direct Sales, Bán Hàng]",VietnamWorks,2025-05-29,2025-05-29T13:24:46.253156



--- Kiểm tra các giá trị bị thiếu (NaN) cho mỗi cột (VietnamWorks) ---
job_title                   0
job_url                     0
company_name                0
salary_raw                  0
location_raw                0
city_main                   0
district                   50
experience                  0
experience_standardized     0
post_date_raw               0
keywords                    0
source_website              0
scrape_date                 0
scraped_at_timestamp        0
dtype: int64

DataFrame 'df_jobs_vnw' đã sẵn sàng cho các bước xử lý chi tiết hơn.


In [6]:
# Cell 6: Detailed Processing for 'salary_raw' column (VietnamWorks data)

df_jobs_vnw = locals().get('df_jobs_vnw', None) # Lấy df_jobs_vnw từ cell trước

if df_jobs_vnw is not None and not df_jobs_vnw.empty:
    if 'salary_raw' not in df_jobs_vnw.columns:
        print("Lỗi: Cột 'salary_raw' không tồn tại trong DataFrame 'df_jobs_vnw'.")
    else:
        print("--- Bắt đầu xử lý cột 'salary_raw' cho VietnamWorks ---")
        
        df_jobs_vnw['salary_min_mil'] = pd.NA 
        df_jobs_vnw['salary_max_mil'] = pd.NA
        df_jobs_vnw['salary_currency'] = None 
        df_jobs_vnw['salary_type'] = None 

        print("\nCác giá trị duy nhất trong cột 'salary_raw' (VNW - trước khi xử lý):")
        if df_jobs_vnw['salary_raw'].notna().any():
            unique_salaries_vnw = df_jobs_vnw['salary_raw'].dropna().unique()
            print(unique_salaries_vnw[:min(20, len(unique_salaries_vnw))])
        else:
            print("Cột 'salary_raw' không có giá trị nào (toàn NaN) hoặc rỗng trong df_jobs_vnw.")

        # --- HÀM parse_salary_string (Logic của bạn đã được cải tiến) ---
        def parse_salary_string(salary_text):
            if pd.isna(salary_text): return [], None, "Không rõ"
            text_input = str(salary_text); text_lower = text_input.lower()
            currency = "triệu VNĐ"; type_guess = "Không xác định"

            if any(kw in text_lower for kw in ["thỏa thuận", "thoả thuận", "thương lượng", "negotiable"]):
                return [], currency, "Thoả thuận"

            if "usd" in text_lower or "$" in text_input:
                currency = "USD"; text_to_extract = re.sub(r'usd|\$', '', text_input, flags=re.IGNORECASE).strip()
            elif "triệu" in text_lower:
                currency = "triệu VNĐ"; text_to_extract = text_lower.replace("triệu", "").replace("vnd","").strip()
            else: text_to_extract = text_lower.replace("vnd","").strip()

            raw_numbers_str = re.findall(r'\d+(?:[.,]\d+)*', text_to_extract)
            numbers = []
            for num_str_raw in raw_numbers_str:
                num_str_cleaned = num_str_raw.replace('.', '').replace(',', '.')
                try: numbers.append(float(num_str_cleaned))
                except ValueError: pass
            numbers.sort()

            if any(kw in text_lower for kw in ["tới", "upto", "đến"]) and "từ" not in text_lower: type_guess = "Tối đa"
            elif "từ" in text_lower and not any(kw in text_lower for kw in ["tới", "upto", "đến"]): type_guess = "Tối thiểu"
            elif (len(numbers) == 2 and '-' in text_to_extract) or \
                 (len(numbers) == 2 and not any(kw in text_lower for kw in ["tới", "upto", "đến", "từ"])): type_guess = "Khoảng"
            elif len(numbers) == 1 and not any(kw in text_lower for kw in ["tới", "upto", "đến", "từ"]): type_guess = "Cố định"
            elif not numbers: type_guess = "Không rõ (Text)"
            return numbers, currency, type_guess
        # --- KẾT THÚC HÀM ---

        for index, row in df_jobs_vnw.iterrows():
            salary_str_original_vnw = row['salary_raw']
            numbers_vnw, currency_detected_vnw, type_guess_vnw = parse_salary_string(salary_str_original_vnw)
            
            df_jobs_vnw.loc[index, 'salary_type'] = type_guess_vnw
            df_jobs_vnw.loc[index, 'salary_currency'] = currency_detected_vnw

            if type_guess_vnw == 'Thoả thuận': pass
            elif type_guess_vnw == 'Tối đa':
                if numbers_vnw: df_jobs_vnw.loc[index, 'salary_max_mil'] = numbers_vnw[-1]
            elif type_guess_vnw == 'Tối thiểu':
                if numbers_vnw: df_jobs_vnw.loc[index, 'salary_min_mil'] = numbers_vnw[0]
            elif type_guess_vnw == 'Khoảng':
                if len(numbers_vnw) == 2:
                    df_jobs_vnw.loc[index, 'salary_min_mil'] = numbers_vnw[0]
                    df_jobs_vnw.loc[index, 'salary_max_mil'] = numbers_vnw[1]
                elif len(numbers_vnw) == 1: 
                    df_jobs_vnw.loc[index, 'salary_min_mil'] = numbers_vnw[0]
                    df_jobs_vnw.loc[index, 'salary_max_mil'] = numbers_vnw[0] 
                    df_jobs_vnw.loc[index, 'salary_type'] = 'Cố định/Lỗi parse khoảng'
            elif type_guess_vnw == 'Cố định':
                if numbers_vnw:
                    df_jobs_vnw.loc[index, 'salary_min_mil'] = numbers_vnw[0]
                    df_jobs_vnw.loc[index, 'salary_max_mil'] = numbers_vnw[0]
        
        df_jobs_vnw['salary_min_mil'] = pd.to_numeric(df_jobs_vnw['salary_min_mil'], errors='coerce')
        df_jobs_vnw['salary_max_mil'] = pd.to_numeric(df_jobs_vnw['salary_max_mil'], errors='coerce')

        print("\n--- DataFrame VietnamWorks sau khi xử lý cột 'salary_raw' ---")
        cols_to_display_salary = ['salary_raw', 'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type']
        existing_cols_salary = [col for col in cols_to_display_salary if col in df_jobs_vnw.columns]
        if existing_cols_salary: display(df_jobs_vnw[existing_cols_salary].head(15))
        
        print("\nKiểm tra kiểu dữ liệu của các cột salary mới (VNW):")
        if 'salary_min_mil' in df_jobs_vnw.columns and 'salary_max_mil' in df_jobs_vnw.columns:
            print(df_jobs_vnw[['salary_min_mil', 'salary_max_mil']].dtypes)
            print("\nThống kê mô tả cho salary_min_mil và salary_max_mil (VNW):")
            display(df_jobs_vnw[['salary_min_mil', 'salary_max_mil']].describe())
        
        print("\nKiểm tra các giá trị 'salary_type' đã phân loại (VNW):")
        if 'salary_type' in df_jobs_vnw.columns: print(df_jobs_vnw['salary_type'].value_counts(dropna=False))
else:
    print("DataFrame 'df_jobs_vnw' không tồn tại hoặc rỗng. Hãy chạy lại Cell 5.")

--- Bắt đầu xử lý cột 'salary_raw' cho VietnamWorks ---

Các giá trị duy nhất trong cột 'salary_raw' (VNW - trước khi xử lý):
['Tới 3tr ₫/tháng' 'Tới $ 140 /tháng' 'Thương lượng' '$ 400-1,500 /tháng'
 '3tr-3tr ₫/tháng' '$ 200-300 /tháng' '3tr-4tr ₫/tháng' '3tr-6tr ₫/tháng'
 '3.42tr-3.6tr ₫/tháng' 'Từ $ 300 /tháng' '8tr-15tr ₫/tháng'
 '$ 100-200 /tháng']

--- DataFrame VietnamWorks sau khi xử lý cột 'salary_raw' ---


,salary_raw,salary_min_mil,salary_max_mil,salary_currency,salary_type
0,Tới 3tr ₫/tháng,NaN,3.0,triệu VNĐ,Tối đa
1,Tới $ 140 /tháng,NaN,140.0,USD,Tối đa
2,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
3,"$ 400-1,500 /tháng",1.5,400.0,USD,Khoảng
4,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
5,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
6,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
7,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
8,Thương lượng,NaN,NaN,triệu VNĐ,Thoả thuận
9,3tr-3tr ₫/tháng,3.0,3.0,triệu VNĐ,Khoảng



Kiểm tra kiểu dữ liệu của các cột salary mới (VNW):
salary_min_mil    float64
salary_max_mil    float64
dtype: object

Thống kê mô tả cho salary_min_mil và salary_max_mil (VNW):


,salary_min_mil,salary_max_mil
count,11.000000,12.000000
mean,77.954545,143.000000
std,107.029074,157.098695
min,1.500000,3.000000
25%,3.000000,3.750000
50%,8.000000,77.500000
75%,150.000000,300.000000
max,300.000000,400.000000



Kiểm tra các giá trị 'salary_type' đã phân loại (VNW):
salary_type
Thoả thuận    37
Khoảng        10
Tối đa         2
Tối thiểu      1
Name: count, dtype: int64


In [7]:
# Cell 7: Detailed Processing for 'post_date_raw' column (VietnamWorks data)

df_jobs_vnw = locals().get('df_jobs_vnw', None)

if df_jobs_vnw is not None and not df_jobs_vnw.empty:
    if 'post_date_raw' not in df_jobs_vnw.columns or 'scrape_date' not in df_jobs_vnw.columns:
        print("Lỗi: Cột 'post_date_raw' hoặc 'scrape_date' không tồn tại trong 'df_jobs_vnw'.")
    else:
        print("--- Bắt đầu xử lý cột 'post_date_raw' cho VietnamWorks ---")
        
        print("\nCác giá trị duy nhất trong 'post_date_raw' (VNW - trước khi xử lý):")
        if df_jobs_vnw['post_date_raw'].notna().any():
            unique_post_dates_vnw = df_jobs_vnw['post_date_raw'].dropna().unique()
            print(unique_post_dates_vnw[:min(20, len(unique_post_dates_vnw))])
        else: print("Cột 'post_date_raw' rỗng hoặc toàn NaN trong df_jobs_vnw.")

        df_jobs_vnw['calculated_post_date'] = pd.NaT
        print(f"\nSử dụng 'scrape_date' làm mốc tính toán cho 'calculated_post_date' (VNW).")

        for index, row in df_jobs_vnw.iterrows():
            date_str_from_scrape_vnw = row['post_date_raw']
            scrape_date_str_from_df_vnw = row['scrape_date'] 
            calculated_date_val_vnw = pd.NaT 

            if pd.isna(date_str_from_scrape_vnw) or pd.isna(scrape_date_str_from_df_vnw):
                df_jobs_vnw.loc[index, 'calculated_post_date'] = pd.NaT; continue

            try:
                scrape_datetime_object_vnw = datetime.strptime(str(scrape_date_str_from_df_vnw), '%Y-%m-%d')
            except ValueError:
                df_jobs_vnw.loc[index, 'calculated_post_date'] = pd.NaT; continue

            date_str_lower_case_vnw = str(date_str_from_scrape_vnw).lower().replace("cập nhật: ", "").strip()

            try:
                if "hôm qua" in date_str_lower_case_vnw: calculated_date_val_vnw = scrape_datetime_object_vnw - timedelta(days=1)
                elif "hôm nay" in date_str_lower_case_vnw: calculated_date_val_vnw = scrape_datetime_object_vnw
                elif "ngày trước" in date_str_lower_case_vnw:
                    days_ago_match = re.search(r'(\d+)\s*ngày trước', date_str_lower_case_vnw)
                    if days_ago_match: days = int(days_ago_match.group(1)); calculated_date_val_vnw = scrape_datetime_object_vnw - timedelta(days=days)
                elif "tuần trước" in date_str_lower_case_vnw:
                    weeks_ago_match = re.search(r'(\d+)\s*tuần trước', date_str_lower_case_vnw)
                    if weeks_ago_match: weeks = int(weeks_ago_match.group(1)); calculated_date_val_vnw = scrape_datetime_object_vnw - timedelta(weeks=weeks)
                elif "tháng trước" in date_str_lower_case_vnw:
                    months_ago_match = re.search(r'(\d+)\s*tháng trước', date_str_lower_case_vnw)
                    if months_ago_match: months = int(months_ago_match.group(1)); calculated_date_val_vnw = scrape_datetime_object_vnw - timedelta(days=int(months * 30.4375))
                else: 
                    try:
                        parsed_directly_vnw = pd.to_datetime(date_str_lower_case_vnw, format='%d/%m/%Y', errors='coerce')
                        if pd.notna(parsed_directly_vnw): calculated_date_val_vnw = parsed_directly_vnw
                        else: 
                             parsed_directly_vnw_alt = pd.to_datetime(date_str_lower_case_vnw, errors='coerce')
                             if pd.notna(parsed_directly_vnw_alt): calculated_date_val_vnw = parsed_directly_vnw_alt
                    except ValueError: pass 
                
                if pd.notna(calculated_date_val_vnw):
                    if isinstance(calculated_date_val_vnw, pd.Timestamp): df_jobs_vnw.loc[index, 'calculated_post_date'] = calculated_date_val_vnw.normalize()
                    elif isinstance(calculated_date_val_vnw, datetime): df_jobs_vnw.loc[index, 'calculated_post_date'] = pd.Timestamp(calculated_date_val_vnw).normalize()
                else: df_jobs_vnw.loc[index, 'calculated_post_date'] = pd.NaT
            except Exception as e_date_proc_vnw:
                df_jobs_vnw.loc[index, 'calculated_post_date'] = pd.NaT

        print("\n--- DataFrame VietnamWorks sau khi xử lý cột 'post_date_raw' ---")
        cols_to_display_post_date_vnw = ['post_date_raw', 'scrape_date', 'calculated_post_date']
        existing_cols_post_date_vnw = [col for col in cols_to_display_post_date_vnw if col in df_jobs_vnw.columns]
        if existing_cols_post_date_vnw: display(df_jobs_vnw[existing_cols_post_date_vnw].head(15))
        
        print("\nKiểm tra kiểu dữ liệu của cột 'calculated_post_date' (VNW):")
        if 'calculated_post_date' in df_jobs_vnw.columns:
            print(df_jobs_vnw['calculated_post_date'].dtype) 
            print("\nKiểm tra các giá trị bị thiếu trong 'calculated_post_date' (VNW):")
            print(f"Số lượng NaT: {df_jobs_vnw['calculated_post_date'].isnull().sum()}")
            print("\nThống kê mô tả cho 'calculated_post_date' (VNW):")
            if df_jobs_vnw['calculated_post_date'].notnull().any():
                display(df_jobs_vnw['calculated_post_date'].describe()) 
            else: print("Không có giá trị 'calculated_post_date' hợp lệ để thống kê (VNW).")
        else: print("Cột 'calculated_post_date' không được tạo cho df_jobs_vnw.")
else:
    print("DataFrame 'df_jobs_vnw' không tồn tại hoặc rỗng. Hãy chạy lại Cell 5.")

--- Bắt đầu xử lý cột 'post_date_raw' cho VietnamWorks ---

Các giá trị duy nhất trong 'post_date_raw' (VNW - trước khi xử lý):
['Hôm nay' 'Cập nhật: 28/05/2025' 'Cập nhật: 27/05/2025'
 'Cập nhật: 26/05/2025' 'Cập nhật: 21/05/2025' 'Cập nhật: 14/05/2025'
 'Cập nhật: 13/05/2025' 'Cập nhật: 09/05/2025' 'Cập nhật: 07/05/2025'
 'Cập nhật: 24/04/2025' 'Cập nhật: 23/04/2025' 'Cập nhật: 22/04/2025'
 'Cập nhật: 23/05/2025' 'Cập nhật: 22/05/2025' 'Cập nhật: 20/05/2025'
 'Cập nhật: 19/05/2025' 'Cập nhật: 16/05/2025' 'Cập nhật: 15/05/2025'
 'Cập nhật: 12/05/2025' 'Cập nhật: 06/05/2025']

Sử dụng 'scrape_date' làm mốc tính toán cho 'calculated_post_date' (VNW).

--- DataFrame VietnamWorks sau khi xử lý cột 'post_date_raw' ---


,post_date_raw,scrape_date,calculated_post_date
0,Hôm nay,2025-05-29,2025-05-29
1,Cập nhật: 28/05/2025,2025-05-29,2025-05-28
2,Cập nhật: 27/05/2025,2025-05-29,2025-05-27
3,Cập nhật: 26/05/2025,2025-05-29,2025-05-26
4,Cập nhật: 26/05/2025,2025-05-29,2025-05-26
5,Cập nhật: 21/05/2025,2025-05-29,2025-05-21
6,Cập nhật: 14/05/2025,2025-05-29,2025-05-14
7,Cập nhật: 13/05/2025,2025-05-29,2025-05-13
8,Cập nhật: 13/05/2025,2025-05-29,2025-05-13
9,Cập nhật: 09/05/2025,2025-05-29,2025-05-09



Kiểm tra kiểu dữ liệu của cột 'calculated_post_date' (VNW):
datetime64[ns]

Kiểm tra các giá trị bị thiếu trong 'calculated_post_date' (VNW):
Số lượng NaT: 0

Thống kê mô tả cho 'calculated_post_date' (VNW):


count                     50
mean     2025-05-18 16:19:12
min      2025-04-22 00:00:00
25%      2025-05-13 00:00:00
50%      2025-05-22 00:00:00
75%      2025-05-27 00:00:00
max      2025-05-29 00:00:00
Name: calculated_post_date, dtype: object

In [8]:
# Cell 8: Review 'experience_standardized' column (VietnamWorks data)

df_jobs_vnw = locals().get('df_jobs_vnw', None)

if df_jobs_vnw is not None and not df_jobs_vnw.empty:
    # Vì 'experience_standardized' đã được gán cứng ở Cell 3 cho VNW,
    # chúng ta chỉ cần kiểm tra và hiển thị nó.
    if 'experience_standardized' in df_jobs_vnw.columns:
        print("--- Xem xét cột 'experience_standardized' cho VietnamWorks ---")
        
        # Hiển thị cả cột 'experience' (gốc, đã gán cứng) và 'experience_standardized'
        cols_to_display_exp_vnw = []
        if 'experience' in df_jobs_vnw.columns: # Cột 'experience' (raw) đã được tạo ở Cell 3
            cols_to_display_exp_vnw.append('experience')
        cols_to_display_exp_vnw.append('experience_standardized')
        
        existing_cols_exp_vnw = [col for col in cols_to_display_exp_vnw if col in df_jobs_vnw.columns]
        if existing_cols_exp_vnw:
             print("\nDataFrame VietnamWorks (cột kinh nghiệm):")
             display(df_jobs_vnw[existing_cols_exp_vnw].head(15))

        print("\nCác giá trị duy nhất trong 'experience_standardized' (VNW):")
        if df_jobs_vnw['experience_standardized'].notna().any():
            print(df_jobs_vnw['experience_standardized'].value_counts(dropna=False))
        else:
            print("Cột 'experience_standardized' không có giá trị (VNW).")
    else:
        print("Lỗi: Cột 'experience_standardized' không tồn tại trong df_jobs_vnw (cần được tạo ở Cell 3).")
else:
    print("DataFrame 'df_jobs_vnw' không tồn tại hoặc rỗng. Hãy chạy lại Cell 5.")

--- Xem xét cột 'experience_standardized' cho VietnamWorks ---

DataFrame VietnamWorks (cột kinh nghiệm):


,experience,experience_standardized
0,Thực tập sinh/Sinh viên,Intern/Fresher
1,Thực tập sinh/Sinh viên,Intern/Fresher
2,Thực tập sinh/Sinh viên,Intern/Fresher
3,Thực tập sinh/Sinh viên,Intern/Fresher
4,Thực tập sinh/Sinh viên,Intern/Fresher
5,Thực tập sinh/Sinh viên,Intern/Fresher
6,Thực tập sinh/Sinh viên,Intern/Fresher
7,Thực tập sinh/Sinh viên,Intern/Fresher
8,Thực tập sinh/Sinh viên,Intern/Fresher
9,Thực tập sinh/Sinh viên,Intern/Fresher



Các giá trị duy nhất trong 'experience_standardized' (VNW):
experience_standardized
Intern/Fresher    50
Name: count, dtype: int64


In [9]:
# Cell 9: Load Processed VietnamWorks Data to PostgreSQL

df_jobs_vnw = locals().get('df_jobs_vnw', None) # Lấy df_jobs_vnw từ các cell trước

if df_jobs_vnw is not None and not df_jobs_vnw.empty:
    load_dotenv() 
    DB_HOST = os.getenv('DB_HOST'); DB_NAME = os.getenv('DB_NAME')
    DB_USER = os.getenv('DB_USER'); DB_PASSWORD = os.getenv('DB_PASSWORD')
    DB_PORT = os.getenv('DB_PORT')

    TABLE_NAME_FOR_VNW = "vnwork_jobs" 

    # SCHEMA CHO BẢNG vnwork_jobs (KHÔNG CÓ experience_raw)
    CREATE_TABLE_FINAL_SQL = f"""
    CREATE TABLE IF NOT EXISTS {TABLE_NAME_FOR_VNW} (
        job_id SERIAL PRIMARY KEY, job_title VARCHAR(500), job_url TEXT UNIQUE,
        company_name VARCHAR(300), salary_raw VARCHAR(100), 
        salary_min_mil NUMERIC(10, 2), salary_max_mil NUMERIC(10, 2),
        salary_currency VARCHAR(20), salary_type VARCHAR(50),
        location_raw TEXT, city_main VARCHAR(100) NULL, district VARCHAR(100) NULL,  
        experience_standardized VARCHAR(50), -- Chỉ cột này cho kinh nghiệm
        post_date_raw VARCHAR(50), calculated_post_date DATE,
        scrape_date DATE, scraped_at_timestamp TIMESTAMP WITHOUT TIME ZONE NULL, 
        source_website VARCHAR(100), keywords TEXT NULL, 
        inserted_at TIMESTAMP WITHOUT TIME ZONE DEFAULT CURRENT_TIMESTAMP 
    );"""
    
    conn_vnw = None; cur_vnw = None  
    try:
        print(f"Đang kết nối đến database '{DB_NAME}' trên host '{DB_HOST}' (cho VNW)...")
        conn_vnw = psycopg2.connect(dbname=DB_NAME, user=DB_USER, password=DB_PASSWORD, host=DB_HOST, port=DB_PORT)
        conn_vnw.autocommit = False; cur_vnw = conn_vnw.cursor() 
        print("Kết nối thành công (VNW)!")
        
        print(f"Đang tạo/kiểm tra bảng '{TABLE_NAME_FOR_VNW}' (VNW)...")
        cur_vnw.execute(CREATE_TABLE_FINAL_SQL)
        print(f"Bảng '{TABLE_NAME_FOR_VNW}' đã sẵn sàng (VNW).")
        
        df_to_insert_vnw = df_jobs_vnw.copy()   
        
        # Cột trong DB để INSERT
        db_columns_for_insert_vnw = [
            'job_title', 'job_url', 'company_name', 'salary_raw', 
            'salary_min_mil', 'salary_max_mil', 'salary_currency', 'salary_type',
            'location_raw', 'city_main', 'district',
            'experience_standardized', # Chỉ cột này
            'post_date_raw', 'calculated_post_date', 'scrape_date',
            'scraped_at_timestamp', 'source_website', 'keywords'
        ]
        
        # Cột tương ứng trong DataFrame df_jobs_vnw để lấy dữ liệu
        # Phải đảm bảo df_jobs_vnw có tất cả các cột này
        dataframe_columns_to_select_vnw = db_columns_for_insert_vnw[:] # Trong trường hợp này, tên cột giống nhau

        missing_cols = [col for col in dataframe_columns_to_select_vnw if col not in df_to_insert_vnw.columns]
        if missing_cols:
            raise KeyError(f"Các cột sau không tìm thấy trong df_jobs_vnw: {missing_cols}")

        list_of_tuples_vnw = []
        for index, row in df_to_insert_vnw.iterrows():
            current_row_tuple_data = []
            for df_col_name in dataframe_columns_to_select_vnw:
                val = row.get(df_col_name)
                if df_col_name == 'keywords' and isinstance(val, list):
                    current_row_tuple_data.append(', '.join(val) if val else None)
                else:
                    current_row_tuple_data.append(val if pd.notna(val) else None)
            list_of_tuples_vnw.append(tuple(current_row_tuple_data))

        if not list_of_tuples_vnw: print("Không có dữ liệu VietnamWorks để chèn.")
        else:
            print(f"Đã chuẩn bị {len(list_of_tuples_vnw)} dòng dữ liệu VietnamWorks để chèn.")
            insert_query_vnw = sql.SQL("INSERT INTO {} ({}) VALUES %s ON CONFLICT (job_url) DO NOTHING;").format(
                sql.Identifier(TABLE_NAME_FOR_VNW), 
                sql.SQL(', ').join(map(sql.Identifier, db_columns_for_insert_vnw))
            )
            print(f"Đang chèn dữ liệu VietnamWorks vào bảng '{TABLE_NAME_FOR_VNW}'...")
            execute_values(cur_vnw, insert_query_vnw.as_string(conn_vnw), list_of_tuples_vnw, page_size=100) 
            inserted_rows_vnw = cur_vnw.rowcount # để lấy số dòng đã chèn
            if inserted_rows_vnw == 0: # để kiểm tra nếu không có dòng nào được chèn
                print("Không có dòng nào được chèn (có thể do trùng lặp job_url).")
            else:
                print(f"Đã chèn {inserted_rows_vnw} dòng dữ liệu VietnamWorks.")
            conn_vnw.commit(); print("Dữ liệu VietnamWorks đã được commit vào database.")
            
    except Exception as e: 
        print(f"Lỗi trong quá trình xử lý DB (VNW): {type(e).__name__} - {e}")
        if conn_vnw: conn_vnw.rollback(); print("Đã rollback (VNW).")
    finally:
        if cur_vnw: cur_vnw.close()
        if conn_vnw: conn_vnw.close(); print("Đã đóng kết nối database (VNW).")
else:
    print("DataFrame 'df_jobs_vnw' không tồn tại hoặc rỗng. Không thể load vào DB.")

Đang kết nối đến database 'my_database' trên host '127.0.0.1' (cho VNW)...
Kết nối thành công (VNW)!
Đang tạo/kiểm tra bảng 'vnwork_jobs' (VNW)...
Bảng 'vnwork_jobs' đã sẵn sàng (VNW).
Đã chuẩn bị 50 dòng dữ liệu VietnamWorks để chèn.
Đang chèn dữ liệu VietnamWorks vào bảng 'vnwork_jobs'...
Không có dòng nào được chèn (có thể do trùng lặp job_url).
Dữ liệu VietnamWorks đã được commit vào database.
Đã đóng kết nối database (VNW).
